# Cloudy Days in Leesburg

This notebook was inspired by the suspicion that, during a certain recent time window, weekends were much cloudier and rainier than weekdays. So I decided to make a proper analysis by downloading weather data and using python/pandas to do some statistical analysis. You should read through this code carefully and figure out what each line of code is doing. When in doubt, use the internet!

## Import pandas

Import pandas with the conventional `pd` alias, then allow tables with many columns to display without truncation.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 300)

# Course helper functions. This downloads ml_utils.py if it isn't already
# here, which is what happens on Colab. Nothing to install.
try:
    import ml_utils
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/paderevski/ML-curriculum/main/ml_utils.py",
        "ml_utils.py")
    import ml_utils

from ml_utils import load_data

Analyze Weather data

## Load the data

Read the weather CSV into a pandas DataFrame named `df`. A DataFrame is a table with labeled rows and columns.

`load_data` fetches the file from the course dataset repository the first time you ask for it, so this works the same on Colab as it does in the lab.

In [ ]:
df = load_data("weather")

## Inspect column names

List the available columns so we know what information the dataset contains and which labels to use in later analyses.

In [ ]:
df.columns

## Summarize numeric data

Use `describe()` to view counts, averages, spread, and extreme values for the numeric columns.

In [ ]:
df.describe()

## Check the table size

Use `shape` to see the number of observations (rows) and variables (columns).

In [ ]:
df.shape

## Count sky conditions

Count each value in the primary sky-condition column. `CLR` means clear conditions.

In [ ]:
df['skyc1'].value_counts()

## View a slice of rows

Use `.loc` to inspect rows 30 through 50 and get a feel for the raw records.

In [ ]:
df.loc[30:50]

## Filter clear observations

Keep only rows where the primary sky condition is `CLR`, then count those rows with `shape`.

In [ ]:
df[df['skyc1']=='CLR'].shape

## Look for an unusual combination

Apply two conditions at once: clear skies and positive precipitation. The `&` operator means both conditions must be true.

In [ ]:
df[(df['skyc1']=='CLR') & (df['p01i']>0)].shape

## Calculate the precipitation rate

Divide the number of observations with precipitation by the total number of observations to estimate its frequency.

In [ ]:
df[df['p01i']>0].shape[0] / df[df['p01i']>=0].shape[0]

## Check for missing sky data

Loop through the sky-condition columns and count the rows where each one is missing (`NaN`).

In [ ]:
for c in ['skyc1', 'skyc2', 'skyc3', 'skyc4']:
	print(c,df[df[c].isna()].shape)

## Compare two sky layers

Filter to rows where the first and second sky-condition layers differ, then display those two columns for clear first-layer observations.

In [ ]:
df[(df['skyc1'] != df['skyc2']) & (df['skyc1']=='CLR')][['skyc1','skyc2']]

## Count sky-layer combinations

Count how often each differing pair of first- and second-layer sky conditions occurs.

In [ ]:
df[(df['skyc1'] != df['skyc2'])][['skyc1','skyc2']].value_counts()

## Total precipitation by weekday

Group the observations by weekday and add the `p01i` precipitation values within each group.

In [ ]:
total_p01i_per_day_et_new = df.groupby('Day Of Week ET')['p01i'].sum()

## Inspect weekday precipitation totals

Display the grouped result to compare total recorded precipitation across days of the week.

In [ ]:
total_p01i_per_day_et_new

We would like to count the number of clear observations per day. We can do this with a `groupby` followed by a `size()` function

## Count clear observations by weekday

Filter to clear observations, group them by weekday, and use `size()` to count observations in each group.

In [ ]:
df[df['skyc1'] == 'CLR'].groupby('Day Of Week ET').size()

## Visualize clear-observation counts

Plot the weekday counts as a bar chart so differences are easier to compare visually.

In [ ]:
df[df['skyc1'] == 'CLR'].groupby('Day Of Week ET').size().plot(kind="bar");

Does every day have the same number of recorded observations?

## Compare total observations

Plot the number of all records per weekday. This reveals whether the clear-sky counts should be converted to percentages.

In [ ]:
df.groupby('Day Of Week ET').size().plot(kind="bar");

## Calculate clear-sky percentages

Divide each weekday's clear-observation count by its total observation count, then multiply by 100.

In [ ]:
percentage_clr_days_et_new = (df[df['skyc1'] == 'CLR'].groupby('Day Of Week ET').size() / df.groupby('Day Of Week ET').size()) * 100

## Inspect the percentages

Display the percentage of observations with clear skies for each weekday.

In [ ]:
percentage_clr_days_et_new

## Visualize clear-sky percentages

Plot the normalized percentages so the comparison is fair even when weekdays have different numbers of records.

In [ ]:
percentage_clr_days_et_new.plot(kind='bar');

## Build a weekday summary table

Combine observation counts, precipitation totals, and cloudy and clear percentages into one DataFrame for comparison.

In [ ]:
# Recalculate the necessary components based on the 'Day Of Week ET' column from the new dataset with daylight information

# Total precipitation (p01i) for each day of the week
total_p01i_per_day_et_new = df.groupby('Day Of Week ET')['p01i'].sum()

# Percentage of OVC and CLR days
percentage_ovc_days_et_new = (df[(df['skyc1'] == 'OVC') | (df['skyc2'] == 'OVC') | (df['skyc3'] == 'OVC')].groupby('Day Of Week ET').size() / df.groupby('Day Of Week ET').size()) * 100
percentage_clr_days_et_new = (df[df['skyc1'] == 'CLR'].groupby('Day Of Week ET').size() / df.groupby('Day Of Week ET').size()) * 100

# Total number of rows (observations) for each day of the week
total_rows_per_day_et_new = df.groupby('Day Of Week ET').size()

# Combine all the data into one DataFrame
combined_et_df_new = pd.DataFrame({
    'Day Of Week ET': total_rows_per_day_et_new.index,
    'Total Number of Rows ET': total_rows_per_day_et_new.values,
    'Total p01i ET': total_p01i_per_day_et_new.values,
    'Percentage of OVC Days ET': percentage_ovc_days_et_new.values,
    'Percentage of CLR Days ET': percentage_clr_days_et_new.values
})

combined_et_df_new

Conclusions? BTW what is day 5 and 6? How else could you define a good/bad day? What ways could you change the question to make your desired outcome more likely? Is this statistically significant?

## Extend the investigation

Use this empty cell to test a revised definition of a good or bad weather day, or to investigate whether the observed differences are meaningful.